19/04/26.

https://www.codewars.com/kata/5ca24526b534ce0018a137b5/train/python

IDEAS.

* Build a dictionary of letters.
* Code rules for how to produce the messages.
    * Do we book capitals into a dictionary, or handle at during processing?

In [1]:
message = "Def Con 1!"

In [2]:
# Split the message up
message.split()

['Def', 'Con', '1!']

In [3]:
for word in message.split():
    print(word)

Def
Con
1!


In [4]:
# Build a button to character dictionary.
button_chars = {"1": ".,?!", "2": "abc", "3": "def", "4": "ghi", "5": "jkl", "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz", "*": "'-+="}
char_buttons = {value: key for key, value in button_chars.items()}

# Build a character to key presses dictionary, starting with lower case letter key presses.
char_keypr_dict = {}
for key, chars in button_chars.items():
    for i, char in enumerate(chars):
        char_keypr_dict[char] = f"{key}" * (i+1)

# Add mumbers and special characters key presses to the dictionary.
for key in button_chars.keys():
    char_keypr_dict[key] = f"{key}-"

char_keypr_dict["1"]

'1-'

In [ ]:
# We now to deal with waiting, so when we have characters in the same button.
# Iterate through each character in the word, and if the previous character belongs to the "same button", then add a space.

word_keypr = []
word = "hihi"

for i, char in enumerate(word):
    # Check if the current and previous character belong to the same button.
    if char_keypr[char][0] == char_keypr[word[i-1]][0]:
        word_keypr.append(f" {char_keypr[char]}")
    else:
         word_keypr.append(f"{char_keypr[char]}")
word_keypr

"".join(word_keypr)

In [ ]:
# Now need to integrate the logic.
# If previous character is a special character, no need to add a space.
# If there is a case switch, no need to add a space at end.
# Logic is getting a little messy, possibly code some previous word flags.
# When there is no case switching: if current and previous have the same button, 

word_keypr = []

word = "DDef"

for i, char in enumerate(word):
    if char.isupper():
        if word[i-1].islower():
            # Case switching needs no hold.
            word_keypr.append(f"#{char_keypr[char.lower()]}")
        else:
            # No case swithcing as upper -> upper.
            # Now if current and previous have same button, if previous is a special character, no space, but if it isn't then insert a space before current.
            # But if current and previous do not have the same button, do not add a space.
            if char_keypr[char.lower()][0] == char_keypr[word[i-1].lower()][0]:
                if word[i-1].lower() in button_chars.keys():
                    word_keypr.append(f"{char_keypr[char.lower()]}")
                else:
                    word_keypr.append(f" {char_keypr[char.lower()]}")
            else:
                word_keypr.append(f"{char_keypr[char.lower()]}")

    else:
        if word[i-1].isupper():
            # Case switching needs no hold.
            word_keypr.append(f"#{char_keypr[char]}")
        else:
            # No case switching as lower -> lower.
            # Now if current and previous have same button, if previous is a special character, no space, but if it isn't then insert a space before current.
            # But if current and previous do not have the same button, do not add a space.
            if char_keypr[char][0] == char_keypr[word[i-1]][0]:
                if word[i-1] in button_chars.keys():
                    word_keypr.append(f"{char_keypr[char]}")
                else:
                    word_keypr.append(f" {char_keypr[char]}")
            else:
                word_keypr.append(f"{char_keypr[char]}")
            
print(word_keypr)

Refinement.

* This logic here took way too long, and there are solid reasons as to why it is the case, as a point of design - you are compiling and doing NLP at the same time. Or, translating a sentence whilst simulataneously learning the rules vs having a glossary, and applying the rules.
* You are performing three if-else checks, one that checks for capitalisation, and then one that checks for it is lower case, then a button check, rolling it all into one.
* Instead, the key word here is *syntactic parsing*, like you did with POS tags for a 10-708 assignment.
* The logic is also not transparent - it takes a long time to read what is going on, and so your code doesn't pass this smell test.
* Key takeaway: Avoid complex if-else structures.

***

* Let's refine.
* Build a dictionary - this is still important to look up key presses.
* Pre-process each character in a word to turn it into a tuple or dictionary that metadata.

In [ ]:
message = "Def Con 1!"
message.split()

In [7]:
word = "A-z"

In [8]:
# tokens = [{}] * len(word) - CR this.
# Add a null, start token, somewhat similar to NLP techniques.

tokens = [{"char": "", "is_cap": False, "is_hold": False, "button": None, "keypress": ""}]
word_keypr = []
for char in word:
    token = {}
    token["char"] = char
    token["is_cap"] = char.isupper()
    token["is_hold"] = char in button_chars.keys()
    token["button"] = char_keypr_dict[char.lower()][0]
    token["keypress"] = char_keypr_dict[char.lower()]
    tokens.append(token)

In [9]:
for i in range(1, len(word) + 1):
    keypress = ""
    prev = tokens[i - 1]
    current = tokens[i]

    if prev["is_cap"] != current["is_cap"]:
        keypress = "".join(["#", keypress])

    elif prev["button"] == current["button"] and not prev["is_hold"]:
        keypress = "".join([" ", keypress])    

    keypress = "".join([keypress, current["keypress"]])
    word_keypr.append(keypress)

In [10]:
word_keypr

['#2', '#**', '9999']

In [ ]:
"".join(word_keypr)dasdsadasd

In [ ]:
message = "A-z"

# Digit button-character dictionary.
button_chars = {"1": ".,?!", "2": "abc", "3": "def", "4": "ghi", "5": "jkl", "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz", "*": "'-+="}

# Character- key presses dictionary, starting with lower case letter key presses.
char_keypr_dict = {}
for key, chars in button_chars.items():
    for i, char in enumerate(chars):
        char_keypr_dict[char] = f"{key}" * (i+1)

# Add mumbers and special characters key presses to the dictionary.
for key in button_chars.keys():
    char_keypr_dict[key] = f"{key}-"

keypr_seq = []

for word in message.split():
    word_keypr = []

    # For each character in a word, add metadata to get an associated character token.
    # Pad the beginning of a word with a null character token.
    tokens = [{"char": "", "is_cap": False, "is_hold": False, "button": None, "keypress": ""}]
    for char in word:
        token = {}
        token["char"] = char
        token["is_cap"] = char.isupper()
        token["is_hold"] = char in button_chars.keys()
        token["button"] = char_keypr_dict[char.lower()][0]
        token["keypress"] = char_keypr_dict[char.lower()]
        tokens.append(token)

    # Given character tokens in a word, create character keypress.
    for i in range(1, len(word) + 1):
        char_keypr = ""
        prev = tokens[i - 1]
        current = tokens[i]

        if prev["is_cap"] != current["is_cap"]:
            char_keypr = "".join(["#", char_keypr])

        elif prev["button"] == current["button"] and not prev["is_hold"]:
            char_keypr = "".join([" ", char_keypr])    

        char_keypr = "".join([char_keypr, current["keypress"]])
        word_keypr.append(char_keypr)

    word_keypr_str = "".join(word_keypr)

    # Add word keypress to keypress sequence.
    keypr_seq.append(word_keypr_str)

In [5]:
keypr_seq

['#2#**9999']

In [6]:
"0".join(keypr_seq)

'#2#**9999'